# Actividad 4 | Aplicar algoritmos de aprendizaje no supervisado con PySpark

**Instituto Tecnológico y de Estudios Superiores de Monterrey**  
**Maestría en Inteligencia Artificial Aplicada**  
**Análisis de grandes volúmenes de datos**  
**Docente:** Dr. Iván Olmos Pineda

* ANGEL VEGA MARTINEZ — A01377304

**Fecha:** 01 de junio de 2026

## 1. Introducción

El aprendizaje no supervisado es una rama del aprendizaje automático en la que los algoritmos trabajan con datos no etiquetados, es decir, con observaciones que no cuentan con una variable objetivo conocida. Su propósito principal es descubrir patrones ocultos, estructuras internas, similitudes o agrupamientos naturales dentro de los datos, sin intervención humana directa en la asignación de categorías. Este enfoque suele emplearse en tareas de clustering,reducción de dimensionalidad y descubrimiento de relaciones entre variables.

Entre los métodos más representativos del aprendizaje no supervisado en la literatura se encuentran los algoritmos de clustering, como K-Means, el clustering jerárquico y los modelos de mezcla gaussiana (Gaussian Mixture Models), los cuales buscan agrupar observaciones similares entre sí y diferentes de las pertenecientes a otros grupos. También destacan técnicas de reducción de dimensionalidad, como el Análisis de Componentes Principales (PCA), que transforma los datos originales en un número menor de componentes conservando la mayor parte de la variabilidad, así como métodos orientados al descubrimiento de estructuras latentes, por ejemplo Latent Dirichlet Allocation (LDA) para modelado de temas.

En el ecosistema de PySpark, la biblioteca MLlib ofrece implementaciones escalables para distintos algoritmos de aprendizaje no supervisado. Dentro de la API recomendada basada en DataFrames, `spark.ml`, se encuentran disponibles algoritmos como KMeans, BisectingKMeans, GaussianMixture, LDA y PowerIterationClustering, además de transformadores útiles para el preprocesamiento y reducción de dimensionalidad, como PCA, `StringIndexer`, `OneHotEncoder`, `VectorAssembler` y `StandardScaler`. La documentación oficial de Spark señala que la API basada en DataFrames es la principal para construir pipelines y trabajar con algoritmos de machine learning de manera uniforme y escalable.

Aunque el experimento principal se diseñó con K-Means, también se probó BisectingKMeans como comparación. Dado que este último obtuvo un valor de silhouette superior, se considera el modelo con mejor desempeño entre los evaluados.

## 2. Configuración del entorno

En esta sección se instala Java y Spark en Google Colab, se configura `findspark` y se crea una `SparkSession` optimizada para ejecución local.

In [ ]:
# Instalación de Java y Spark en Google Colab
!apt-get update -qq
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz
!tar -xzf spark-3.5.8-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.8-bin-hadoop3"

import findspark
findspark.init()

from pyspark.sql import SparkSession

# Configuración optimizada para Colab/local mode
spark = SparkSession.builder     .master("local[*]")     .appName("PySpark_Optimizado_IncidentGrade")     .config("spark.sql.shuffle.partitions", "8")     .config("spark.default.parallelism", "8")     .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")     .config("spark.driver.memory", "8g")     .config("spark.sql.execution.arrow.pyspark.enabled", "true")     .getOrCreate()

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
print(spark.version)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
3.5.8


## 3. Parámetros del experimento

Estos parámetros permiten ajustar rápidamente el tamaño de la muestra, la reducción de cardinalidad y los valores de `k` que serán probados en K-Means.

In [4]:
# ===============================
# Parámetros rápidos de ajuste
# ===============================
SAMPLE_FRAC = 0.02
TOP_K_CATEGORY = 10         # top-k categorías a conservar en CategoryReduced
SEED = 42

# Valores candidatos para KMeans
K_VALUES = [2, 3, 4, 5, 6]
KMEANS_MAX_ITER = 20

# Para comparar con BisectingKMeans
RUN_BISECTING = True

# Mostrar salidas principales
SHOW_CLUSTER_SIZES = True
SHOW_CLUSTER_COMPOSITION = True
SHOW_CLUSTER_CENTERS = False

## 4. Carga de datos desde Google Drive

Se utiliza la misma base del proyecto y el mismo `schema` explícito que ya se construyó en la actividad de aprendizaje supervisado, lo cual hace que la lectura sea más rápida y consistente.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

file_path = "/content/drive/MyDrive/Colab Notebooks/Análisis de grandes volúmenes de datos (Gpo 10)/Microsoft_GUIDE_Train.csv"

Mounted at /content/drive


In [6]:
from pyspark.sql.types import (
    StructType, StructField, LongType, IntegerType, DoubleType, StringType, TimestampType
)

schema = StructType([
    StructField("Id", LongType(), True),
    StructField("OrgId", IntegerType(), True),
    StructField("IncidentId", IntegerType(), True),
    StructField("AlertId", IntegerType(), True),
    StructField("Timestamp", TimestampType(), True),
    StructField("DetectorId", IntegerType(), True),
    StructField("AlertTitle", IntegerType(), True),
    StructField("Category", StringType(), True),
    StructField("MitreTechniques", StringType(), True),
    StructField("IncidentGrade", StringType(), True),
    StructField("ActionGrouped", StringType(), True),
    StructField("ActionGranular", StringType(), True),
    StructField("EntityType", StringType(), True),
    StructField("EvidenceRole", StringType(), True),
    StructField("DeviceId", IntegerType(), True),
    StructField("Sha256", IntegerType(), True),
    StructField("IpAddress", IntegerType(), True),
    StructField("Url", IntegerType(), True),
    StructField("AccountSid", IntegerType(), True),
    StructField("AccountUpn", IntegerType(), True),
    StructField("AccountObjectId", IntegerType(), True),
    StructField("AccountName", IntegerType(), True),
    StructField("DeviceName", IntegerType(), True),
    StructField("NetworkMessageId", IntegerType(), True),
    StructField("EmailClusterId", DoubleType(), True),
    StructField("RegistryKey", IntegerType(), True),
    StructField("RegistryValueName", IntegerType(), True),
    StructField("RegistryValueData", IntegerType(), True),
    StructField("ApplicationId", IntegerType(), True),
    StructField("ApplicationName", IntegerType(), True),
    StructField("OAuthApplicationId", IntegerType(), True),
    StructField("ThreatFamily", StringType(), True),
    StructField("FileName", IntegerType(), True),
    StructField("FolderPath", IntegerType(), True),
    StructField("ResourceIdName", IntegerType(), True),
    StructField("ResourceType", StringType(), True),
    StructField("Roles", StringType(), True),
    StructField("OSFamily", IntegerType(), True),
    StructField("OSVersion", IntegerType(), True),
    StructField("AntispamDirection", StringType(), True),
    StructField("SuspicionLevel", StringType(), True),
    StructField("LastVerdict", StringType(), True),
    StructField("CountryCode", IntegerType(), True),
    StructField("State", IntegerType(), True),
    StructField("City", IntegerType(), True),
    StructField("CategoryGroup", StringType(), True),
    StructField("EntityTypeGroup", StringType(), False)
])

df = spark.read     .option("header", True)     .schema(schema)     .csv(file_path)

print("Columnas cargadas:", len(df.columns))
df.printSchema()

Columnas cargadas: 47
root
 |-- Id: long (nullable = true)
 |-- OrgId: integer (nullable = true)
 |-- IncidentId: integer (nullable = true)
 |-- AlertId: integer (nullable = true)
 |-- Timestamp: timestamp (nullable = true)
 |-- DetectorId: integer (nullable = true)
 |-- AlertTitle: integer (nullable = true)
 |-- Category: string (nullable = true)
 |-- MitreTechniques: string (nullable = true)
 |-- IncidentGrade: string (nullable = true)
 |-- ActionGrouped: string (nullable = true)
 |-- ActionGranular: string (nullable = true)
 |-- EntityType: string (nullable = true)
 |-- EvidenceRole: string (nullable = true)
 |-- DeviceId: integer (nullable = true)
 |-- Sha256: integer (nullable = true)
 |-- IpAddress: integer (nullable = true)
 |-- Url: integer (nullable = true)
 |-- AccountSid: integer (nullable = true)
 |-- AccountUpn: integer (nullable = true)
 |-- AccountObjectId: integer (nullable = true)
 |-- AccountName: integer (nullable = true)
 |-- DeviceName: integer (nullable = true)
 |

## 5. Selección de los datos y construcción de la muestra M'

A partir de la base original se construye una muestra reducida **M'**. Esta sección busca cumplir con el criterio de la rúbrica relacionado con la **construcción de la muestra**:

1. Se parte de la base global del proyecto.
2. Se filtran registros válidos.
3. Se reduce la cardinalidad de `Category`.
4. Se generan variables temporales a partir de `Timestamp`.
5. Se seleccionan únicamente variables con potencial informativo y costo computacional moderado.
6. Se construye **M'** mediante muestreo estratificado usando `IncidentGrade` solo para conservar representatividad de la muestra.

> **Importante:** aunque `IncidentGrade` se conserva en la muestra, **no será usada como variable objetivo ni como feature** durante el entrenamiento no supervisado. Se mantendrá únicamente para interpretar los clusters al final.

In [7]:
from pyspark.sql.functions import col, when, hour, dayofweek, month, desc

# Filtrar registros válidos; conservamos IncidentGrade solo para análisis posterior
base = df.filter(col("IncidentGrade").isNotNull())

# Reducir cardinalidad de Category utilizando las TOP_K_CATEGORY categorías más frecuentes
top_categories = [
    row["Category"]
    for row in base.groupBy("Category")
        .count()
        .orderBy(desc("count"))
        .limit(TOP_K_CATEGORY)
        .collect()
]

base = base.withColumn(
    "CategoryReduced",
    when(col("Category").isin(top_categories), col("Category")).otherwise("OtherCategory")
)

# Variables temporales básicas a partir de Timestamp
base = base.withColumn("Hour", hour(col("Timestamp")))            .withColumn("DayOfWeek", dayofweek(col("Timestamp")))            .withColumn("Month", month(col("Timestamp")))

# Selección compacta de columnas (alineada con la actividad supervisada, pero sin usar una variable objetivo)
selected_cols = [
    "IncidentId",
    "IncidentGrade",      # solo para interpretación de clusters
    "OrgId",
    "DetectorId",
    "CategoryReduced",
    "ActionGrouped",
    "EntityType",
    "EvidenceRole",
    "ResourceType",
    "SuspicionLevel",
    "LastVerdict",
    "OSFamily",
    "OSVersion",
    "CountryCode",
    "CategoryGroup",
    "EntityTypeGroup",
    "Hour",
    "DayOfWeek",
    "Month"
]

base = base.select(*selected_cols).dropDuplicates()

# Construcción de la muestra M' mediante muestreo estratificado por IncidentGrade
# Esta variable NO se usará para entrenar el modelo; solo ayuda a preservar la representatividad de la muestra.
fractions = {
    row["IncidentGrade"]: SAMPLE_FRAC
    for row in base.select("IncidentGrade").distinct().collect()
}

df_muestra = base.sampleBy("IncidentGrade", fractions=fractions, seed=SEED)

df_muestra = df_muestra.cache()
_ = df_muestra.count()

print("Muestra M' lista.")
print("Filas en M':", df_muestra.count())
df_muestra.groupBy("IncidentGrade").count().orderBy(desc("count")).show()

Muestra M' lista.
Filas en M': 79050
+--------------+-----+
| IncidentGrade|count|
+--------------+-----+
|BenignPositive|35688|
|  TruePositive|24010|
| FalsePositive|19352|
+--------------+-----+



## 6. Preparación del conjunto de entrenamiento y prueba

Aunque el aprendizaje no supervisado no utiliza etiquetas, en esta actividad se construyen conjuntos de **entrenamiento y prueba** para evaluar la estabilidad del agrupamiento en datos no vistos. La partición se realiza con una proporción aproximada de **80/20** y, para minimizar el riesgo de sesgo y fuga de información, se realiza con base en `IncidentId`. De este modo, registros asociados al mismo incidente no aparecen simultáneamente en ambos conjuntos.

In [8]:
from pyspark.sql.functions import rand

incident_ids = df_muestra.select("IncidentId").distinct().withColumn("rand", rand(seed=SEED))

train_ids = incident_ids.filter(col("rand") < 0.8).select("IncidentId")
test_ids  = incident_ids.filter(col("rand") >= 0.8).select("IncidentId")

train_df = df_muestra.join(train_ids, on="IncidentId", how="inner")
test_df  = df_muestra.join(test_ids, on="IncidentId", how="inner")

train_df = train_df.cache()
test_df = test_df.cache()
_ = train_df.count()
_ = test_df.count()

print("Train:", train_df.count())
print("Test:", test_df.count())

Train: 63207
Test: 15843


## 7. Tratamiento de nulos y definición de variables

Para el clustering se utilizarán variables numéricas y categóricas de costo computacional moderado. Como K-Means se basa en distancias, no es adecuado usar directamente categorías como números enteros sin preparación adicional; por ello, más adelante dichas variables se transformarán mediante `StringIndexer` y `OneHotEncoder`.

In [9]:
categorical_cols = [
    "CategoryReduced",
    "ActionGrouped",
    "EntityType",
    "EvidenceRole",
    "ResourceType",
    "SuspicionLevel",
    "LastVerdict",
    "CategoryGroup",
    "EntityTypeGroup"
]

numeric_cols = [
    "OrgId",
    "DetectorId",
    "OSFamily",
    "OSVersion",
    "CountryCode",
    "Hour",
    "DayOfWeek",
    "Month"
]

# Tratamiento simple de nulos
train_df = train_df.fillna("Desconocido", subset=categorical_cols)
test_df = test_df.fillna("Desconocido", subset=categorical_cols)

for c in numeric_cols:
    train_df = train_df.fillna({c: 0})
    test_df = test_df.fillna({c: 0})

print("Nulos tratados correctamente.")

Nulos tratados correctamente.


## 8. Preparación de features para clustering

Debido a que K-Means utiliza una medida de distancia, se necesita una preparación cuidadosa de las variables de entrada:

1. Las variables categóricas se transforman con `StringIndexer`.
2. Luego se codifican con `OneHotEncoder`, para evitar imponer un orden artificial a las categorías.
3. Se ensamblan todas las variables en una sola columna vectorial con `VectorAssembler`.
4. Finalmente, las features se escalan con `StandardScaler`, favoreciendo que las variables numéricas no dominen las distancias solo por su magnitud.

In [10]:
import time
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

# 1. Indexación de categóricas
feature_indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]

# 2. One-Hot Encoding
encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_cols],
    outputCols=[f"{c}_ohe" for c in categorical_cols],
    handleInvalid="keep"
)

# 3. Ensamblar numéricas + categóricas codificadas
assembler_raw = VectorAssembler(
    inputCols=numeric_cols + [f"{c}_ohe" for c in categorical_cols],
    outputCol="raw_features",
    handleInvalid="keep"
)

# 4. Escalar features para favorecer el uso de distancias en KMeans
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=False
)

prep_pipeline = Pipeline(stages=feature_indexers + [encoder, assembler_raw, scaler])

start = time.perf_counter()
prep_model = prep_pipeline.fit(train_df)

train_prepared = prep_model.transform(train_df).select("IncidentId", "IncidentGrade", "features")
test_prepared  = prep_model.transform(test_df).select("IncidentId", "IncidentGrade", "features")

train_prepared = train_prepared.cache()
test_prepared = test_prepared.cache()
_ = train_prepared.count()
_ = test_prepared.count()

prep_time = time.perf_counter() - start
print(f"Preprocesamiento completado en {prep_time/60:.2f} minutos")
print("Filas train preparadas:", train_prepared.count())
print("Filas test preparadas:", test_prepared.count())

Preprocesamiento completado en 0.33 minutos
Filas train preparadas: 63207
Filas test preparadas: 15843


## 9. Construcción del modelo no supervisado: K-Means

En esta sección se prueba el algoritmo **K-Means** para distintos valores de `k`, usando la métrica **silhouette** como criterio básico de calidad. El mejor valor de `k` se seleccionará con base en el resultado sobre el conjunto de prueba.

In [11]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

results = []

evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="prediction",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

for k in K_VALUES:
    start = time.perf_counter()

    kmeans = KMeans(
        k=k,
        seed=SEED,
        featuresCol="features",
        predictionCol="prediction",
        maxIter=KMEANS_MAX_ITER,
        initMode="k-means||"
    )

    model_k = kmeans.fit(train_prepared)
    pred_train = model_k.transform(train_prepared)
    pred_test = model_k.transform(test_prepared)

    silhouette_train = evaluator.evaluate(pred_train)
    silhouette_test = evaluator.evaluate(pred_test)
    elapsed = time.perf_counter() - start

    results.append((int(k), float(silhouette_train), float(silhouette_test), float(elapsed)))

print("Resultados de KMeans por valor de k:")
for k, s_train, s_test, elapsed in results:
    print(f"k={k} | silhouette_train={s_train:.4f} | silhouette_test={s_test:.4f} | tiempo={elapsed/60:.2f} min")

Resultados de KMeans por valor de k:
k=2 | silhouette_train=0.1895 | silhouette_test=0.2175 | tiempo=0.23 min
k=3 | silhouette_train=-0.1163 | silhouette_test=-0.1102 | tiempo=0.18 min
k=4 | silhouette_train=-0.0685 | silhouette_test=0.0063 | tiempo=0.13 min
k=5 | silhouette_train=-0.0585 | silhouette_test=-0.0538 | tiempo=0.16 min
k=6 | silhouette_train=-0.0451 | silhouette_test=-0.0284 | tiempo=0.13 min


In [12]:
# Seleccionar el mejor k con base en silhouette_test
best_row = max(results, key=lambda x: x[2])
best_k = best_row[0]

print(f"Mejor k seleccionado: {best_k}")
print(f"Silhouette en test del mejor k: {best_row[2]:.4f}")

Mejor k seleccionado: 2
Silhouette en test del mejor k: 0.2175


## 10. Entrenamiento del modelo final y evaluación

Una vez seleccionado el mejor valor de `k`, se reentrena K-Means sobre el conjunto de entrenamiento y se evalúa sobre el conjunto de prueba mediante la métrica silhouette. También se revisan los tamaños de los clusters y su composición con respecto a `IncidentGrade`, utilizada **solo para análisis interpretativo**.

In [13]:
# Modelo final con el mejor k
kmeans_final = KMeans(
    k=best_k,
    seed=SEED,
    featuresCol="features",
    predictionCol="prediction",
    maxIter=KMEANS_MAX_ITER,
    initMode="k-means||"
)

final_model = kmeans_final.fit(train_prepared)
final_train_pred = final_model.transform(train_prepared).cache()
final_test_pred = final_model.transform(test_prepared).cache()
_ = final_train_pred.count()
_ = final_test_pred.count()

final_silhouette_train = evaluator.evaluate(final_train_pred)
final_silhouette_test = evaluator.evaluate(final_test_pred)

print(f"Silhouette final (train): {final_silhouette_train:.4f}")
print(f"Silhouette final (test): {final_silhouette_test:.4f}")

Silhouette final (train): 0.1895
Silhouette final (test): 0.2175


In [14]:
if SHOW_CLUSTER_SIZES:
    print("Tamaño de clusters en test:")
    final_test_pred.groupBy("prediction").count().orderBy("prediction").show()

Tamaño de clusters en test:
+----------+-----+
|prediction|count|
+----------+-----+
|         0|12356|
|         1| 3487|
+----------+-----+



In [15]:
if SHOW_CLUSTER_COMPOSITION:
    print("Composición de clusters en test respecto a IncidentGrade (solo interpretación ex post):")
    final_test_pred.groupBy("prediction", "IncidentGrade")         .count()         .orderBy("prediction", "IncidentGrade")         .show(100, truncate=False)

Composición de clusters en test respecto a IncidentGrade (solo interpretación ex post):
+----------+--------------+-----+
|prediction|IncidentGrade |count|
+----------+--------------+-----+
|0         |BenignPositive|4899 |
|0         |FalsePositive |3545 |
|0         |TruePositive  |3912 |
|1         |BenignPositive|2281 |
|1         |FalsePositive |404  |
|1         |TruePositive  |802  |
+----------+--------------+-----+



In [18]:
if SHOW_CLUSTER_CENTERS:
    print("Centros de cluster del modelo final:")
    for i, center in enumerate(final_model.clusterCenters()):
        print(f"Cluster {i}: {center}")

## 11. Comparación: BisectingKMeans

Se utiliza el mismo valor de `k` para contrastar la métrica silhouette sobre el conjunto de prueba.

In [17]:
if RUN_BISECTING:
    from pyspark.ml.clustering import BisectingKMeans

    bkm = BisectingKMeans(
        k=best_k,
        seed=SEED,
        featuresCol="features",
        predictionCol="prediction",
        maxIter=KMEANS_MAX_ITER
    )

    bkm_model = bkm.fit(train_prepared)
    bkm_test_pred = bkm_model.transform(test_prepared).cache()
    _ = bkm_test_pred.count()

    bkm_silhouette = evaluator.evaluate(bkm_test_pred)
    print(f"Silhouette BisectingKMeans (test): {bkm_silhouette:.4f}")

Silhouette BisectingKMeans (test): 0.2468


## 12. Interpretación de resultados

## Resultados e interpretación

Para el experimento principal de clustering se utilizó el algoritmo `KMeans`, evaluando distintos valores de `k` mediante la métrica **silhouette**. El mejor resultado se obtuvo con **k = 2**, alcanzando un valor de **silhouette de 0.2175** en el conjunto de prueba.

La métrica silhouette toma valores entre -1 y 1. Valores cercanos a 1 indican clusters compactos y bien separados, mientras que valores cercanos a 0 reflejan una separación débil y valores negativos sugieren asignaciones poco consistentes. En este caso, el valor obtenido indica que los datos presentan una **estructura de agrupamiento existente pero moderada**, es decir, el modelo identifica cierta organización interna, aunque la separación entre clusters no es particularmente fuerte.

En el conjunto de prueba, K-Means generó dos clusters con tamaños distintos: el **cluster 0** quedó conformado por 12,356 registros y el **cluster 1** por 3,487 registros. Este resultado sugiere que el algoritmo encontró un grupo principal más amplio y otro grupo más pequeño y específico.

Para interpretar mejor estos clusters, se analizó su composición con respecto a la variable `IncidentGrade`, utilizada únicamente de forma posterior al entrenamiento. En el **cluster 0**, la distribución fue relativamente heterogénea: 39.65% de los registros correspondieron a `BenignPositive`, 31.66% a `TruePositive` y 28.69% a `FalsePositive`. En cambio, el **cluster 1** mostró una mayor concentración de incidentes `BenignPositive`, con 65.42% de sus registros, mientras que `TruePositive` representó 23.00% y `FalsePositive` 11.59%.

Estos resultados sugieren que K-Means logró separar un grupo con mayor concentración de incidentes benignos, mientras que el otro cluster agrupa observaciones más mixtas y menos diferenciadas. En consecuencia, puede afirmarse que el modelo no supervisado fue capaz de detectar cierta estructura interna relevante en los datos, aunque la capacidad de separación entre grupos todavía es moderada.

Como comparación adicional, se entrenó un modelo con `BisectingKMeans`, obteniendo un valor de **silhouette de 0.2468** en el conjunto de prueba. Este resultado fue superior al de K-Means, lo cual indica que `BisectingKMeans` generó una partición ligeramente más consistente y mejor separada para esta muestra. Por ello, dentro de los modelos evaluados, `BisectingKMeans` puede considerarse la alternativa con mejor desempeño en este experimento.

## 13. Conclusiones

En esta actividad se implementó un experimento de aprendizaje no supervisado en PySpark sobre una muestra reducida de la base del proyecto. Se documentaron las etapas de selección de los datos, construcción de la muestra M’, preparación del conjunto de entrenamiento y prueba, transformación de variables y entrenamiento de modelos de clustering.

La muestra final M’ quedó conformada por 79,050 registros, y posteriormente fue dividida en un conjunto de entrenamiento y otro de prueba con una proporción aproximada de 80/20, utilizando `IncidentId` como criterio de partición para reducir el riesgo de fuga de información. Después de preparar las variables categóricas y numéricas mediante indexación, codificación one-hot, ensamblado y escalado, se aplicó el algoritmo `KMeans` para distintos valores de `k`, seleccionando como mejor configuración `k = 2`.

El modelo K-Means alcanzó un valor de **silhouette de 0.2175** en el conjunto de prueba, lo cual indica una estructura de agrupamiento existente pero moderada. El análisis de la composición de los clusters mostró que uno de los grupos concentró una mayor proporción de incidentes `BenignPositive`, mientras que el otro agrupó observaciones más heterogéneas.

Finalmente, al comparar con `BisectingKMeans`, se obtuvo un mejor resultado, con una **silhouette de 0.2468**, lo que sugiere que este algoritmo produjo una partición ligeramente más consistente y mejor separada. En consecuencia, se concluye que PySpark permite implementar de forma escalable modelos de aprendizaje no supervisado sobre grandes volúmenes de datos y que, dentro de las alternativas evaluadas en este trabajo, `BisectingKMeans` fue el modelo de mejor desempeño.